# Component 1 Notebook

This notebook is a thin interface over the Component 1 Python modules.
The actual implementation lives in `src/components/component1_encoder.py`, `src/components/component1_dann.py`, and `src/training/train_component1_dann.py`.

Use this notebook to inspect the HDD dataset layout, run a forward pass, and launch training.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

PosixPath('/Users/ahmedhabib/Documents/Dl/dl-project-codebase')

In [2]:
from collections import Counter
from pprint import pprint
import subprocess

import matplotlib.pyplot as plt
import torch

from src.training.train_component1_dann import (
    Component1DomainDataset,
    build_component1_manifest,
    build_model,
    load_yaml_config,
)


## Load Config

The default paths already point to your external HDD layout shown in Finder.

In [3]:
PATHS_CONFIG = REPO_ROOT / "configs" / "paths.yaml"
COMPONENT1_CONFIG = REPO_ROOT / "configs" / "component1_dann.yaml"

paths_cfg = load_yaml_config(PATHS_CONFIG)
component1_cfg = load_yaml_config(COMPONENT1_CONFIG)["component1_dann"]

print("encoder backend:", component1_cfg["encoder"]["backend"])
print("checkpoint path:", component1_cfg["encoder"]["checkpoint_path"])
pprint(paths_cfg["datasets"])


encoder backend: auto
checkpoint path: /Volumes/Toshiba HDR/TB_DATA/checkpoints_mirror/sam_vit_h_4b8939.pth
{'montgomery': '/Volumes/Toshiba HDR/TB_DATA/raw/montgomery',
 'nih_cxr14': '/Volumes/Toshiba HDR/TB_DATA/raw/nih_cxr14',
 'shenzhen': '/Volumes/Toshiba HDR/TB_DATA/raw/shenzhen',
 'tbx11k': '/Volumes/Toshiba HDR/TB_DATA/raw/tbx11k'}


## Build Domain Manifest

This scans Montgomery, Shenzhen, TBX11K, and NIH ChestX-ray14. NIH images are indexed from the `images_*.tar.gz` archives using `train_val_list.txt`.

In [4]:
manifest = build_component1_manifest(paths_config=PATHS_CONFIG, component1_config=COMPONENT1_CONFIG)
counts = Counter(sample.dataset_id for sample in manifest)
counts, len(manifest)


KeyboardInterrupt: 

## Inspect One Sample

This uses the same dataset loader as the training script, so it exercises the real HDD-reading path.

In [ ]:
dataset = Component1DomainDataset(manifest)
sample = dataset[0]

print("dataset_id:", sample["dataset_id"])
print("source:", sample["source"])
print("x_3ch:", tuple(sample["x_3ch"].shape), sample["x_3ch"].dtype)
print("domain_id:", int(sample["domain_id"]))


In [ ]:
plt.figure(figsize=(5, 5))
plt.imshow(sample["x_3ch"][0].numpy(), cmap="gray")
plt.title(f"{sample['dataset_id']} | {sample['source']}")
plt.axis("off")
plt.show()


## Forward Pass Through Component 1

The config now uses an `auto` backend. If `segment_anything` is installed and the real `sam_vit_h_4b8939.pth` checkpoint exists at the configured HDD path, the notebook will use real SAM ViT-H automatically; otherwise it falls back to the mock contract-preserving encoder.

In [ ]:
model = build_model(component1_cfg)
model.eval()

with torch.no_grad():
    img_emb, dom_logits = model(sample["x_3ch"].unsqueeze(0), lambda_=0.0)
    dom_probs = torch.softmax(dom_logits, dim=1)

print("img_emb:", tuple(img_emb.shape))
print("dom_logits:", tuple(dom_logits.shape))
print("dom_probs:", dom_probs.squeeze(0).tolist())


## Launch Training

Use dry-run first to verify dataset counts. Then disable `DRY_RUN` to start training.

In [ ]:
DRY_RUN = False

cmd = [
    sys.executable,
    "-m",
    "src.training.train_component1_dann",
]
if DRY_RUN:
    cmd.append("--dry-run")

result = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"command failed with exit code {result.returncode}")
